# 8 dang MOI (D14, D28, D32, D33, D34, D40, D42, D49) - FULL90 - ZERO-SHOT - DeepSeek-R1-Distill-Qwen-1.5B

Dung DUNG PROMPT_TEMPLATE tieng Viet va cach cham diem da xai o ban Qwen (`KLTN_D53_8B_zeroshot_full90.ipynb`) - khong doi 1 chu nao, chi doi ten model va nguon du lieu.

`temperature=0.6, top_p=0.95` (dung y het gia tri ban Qwen3-8B zeroshot da dung, cung la khuyen nghi chinh thuc DeepSeek cho dong R1-Distill).

`DATA_DIR` giu NGUYEN `/kaggle/input/soict-11dang-full90` - dung CHUNG dataset Kaggle da co san cho 2 file Nhom/ban ReAct 8dangMoi, khong can upload gi moi.

## Ban Llama-3.2-3B-Instruct - test cheo model

`model=unsloth/Llama-3.2-3B-Instruct`, `temperature=0.6, top_p=0.9` (default CHINH THUC cua Meta - trich tu generation.py chinh thuc trong repo GitHub meta-llama; da xac nhan Meta KHONG cong bo range 'tuan thu vs sang tao' nao ca nen dung thang default, khong tu giam nua). Prompt tieng Viet giu NGUYEN, KHONG dich - moi thu khac (cau truc merge, dataset, forced-prefix, PART_TOOL/KIENTHUC/HUONGGIAI/FEWSHOT/NHIEMVU tung dang) giu NGUYEN 100% Y CHANG ban Qwen3-4B goc da verified.

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

## Dang D14 - zero-shot

In [ ]:
import os, json, re, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d14_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D14']
print('So cau:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D14, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D28 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d28_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D28']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D28, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D32 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d32_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D32']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D32, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D33 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d33_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D33']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D33, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D34 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d34_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D34']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D34, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D40 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d40_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D40']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D40, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D42 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d42_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D42']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D42, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])


## Dang D49 - zero-shot

In [ ]:
MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/8-dang-full/plan_solve_prompts_8dangMoi_merged.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d49_zeroshot_full90_Llama-3.2-3B-Instruct.csv'

MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.9, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D49']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) lan 2 tung gay loi GPU het bo nho (da xac nhan qua
# thuc te chay tren Kaggle).
print('Dung lai tok/llm da nap san.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''

prompts = []
for r in records:
    txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': txt}],
        tokenize=False, add_generation_prompt=True))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f'Sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

rows = []
for r, out in zip(records, outputs):
    txt = out.outputs[0].text
    ntok = len(out.outputs[0].token_ids)
    think = txt.split('</think>')[0] if '</think>' in txt else txt
    after = txt.split('</think>')[-1] if '</think>' in txt else ''
    het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1

    chon = get_letter(after if after else txt)  # FIX: Llama khong co </think>, after luon rong

    rows.append({
        'STT': r['STT'], 'loai_so': r['loai_so'],
        'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
        'DUNG': chon == r['dap_an_letter'],
        'token': ntok, 'het_ngan_sach': het_ngan_sach,
        'think': think, 'after': after,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('='*70)
print('KET QUA TONG (ZERO-SHOT, D49, Llama-3.2-3B-Instruct):', df['DUNG'].sum(), '/', len(df), 'DUNG',
      f"({df['DUNG'].mean()*100:.1f}%)")
print('='*70)
print()
print('--- Theo loai_so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Cac cau SAI ---')
print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon', 'het_ngan_sach']])
